In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [ ]:
# --- 1. LOAD BASE DATA ---
# Pastikan membaca file csv sebagai string (varchar) untuk menghindari error tipe data
users_df = pd.read_csv('users.csv', dtype=str)
chapters_df = pd.read_csv('chapters.csv', dtype=str)
subchapter_df = pd.read_csv('subchapter.csv', dtype=str)

# Ambil list ID yang valid
student_ids = users_df['StudentID'].dropna().tolist()
chapter_ids = chapters_df['id'].dropna().tolist()
subchapter_ids = subchapter_df['id'].dropna().tolist()

# Fungsi untuk generate tanggal random
def random_date(start_year=2024):
    start = datetime(start_year, 1, 1)
    end = datetime(2025, 4, 1)
    return start + timedelta(seconds=random.randint(0, int((end - start).total_seconds())))

# --- 2. GENERATE CHAPTERS TAKEN ---
n_ct = 1200
ct_data = []
for i in range(1, n_ct + 1):
    c_at = random_date()
    u_at = c_at + timedelta(hours=random.randint(1, 48))
    ct_data.append({
        'id': f'CHT{str(i).zfill(5)}', # CHT00001
        'user_id': random.choice(student_ids),
        'chapter_id': random.choice(chapter_ids), # Menggunakan ID baru dari chapters.csv
        'created_at': c_at,
        'updated_at': u_at
    })

df_ct = pd.DataFrame(ct_data)

# Bikin Kotor: Duplikat user yang mengambil chapter yang sama
duplicates_ct = df_ct.sample(20).copy()
df_ct = pd.concat([df_ct, duplicates_ct], ignore_index=True)


# --- 3. GENERATE ASSESSMENTS ---
n_as = 200
as_data = []
types = ['placement', 'quiz', 'exam']
cht_ids = df_ct['id'].tolist()

for i in range(1, n_as + 1):
    c_at = random_date()
    as_data.append({
        'id': f'AS{str(i).zfill(5)}', # AS00001
        'chapter_taken_id': random.choice(cht_ids),
        'subchapter_id': random.choice(subchapter_ids), # Menggunakan ID baru SCT00001
        'title': f'Assessment {random.choice(["A", "B", "C"])} - {i}',
        'type': random.choice(types),
        'correct_answer': random.randint(5, 25),
        'created_at': c_at,
        'updated_at': c_at + timedelta(minutes=random.randint(5, 60))
    })

df_as = pd.DataFrame(as_data)

# Bikin Kotor: Typo di enum type dan null values
df_as.loc[df_as.sample(frac=0.05).index, 'type'] = 'quiz_test'
df_as.loc[df_as.sample(frac=0.05).index, 'correct_answer'] = np.nan


# --- 4. GENERATE USER ATTEMPTS ---
n_ua = 2000
ua_data = []
assessment_list = df_as['id'].tolist()

for i in range(1, n_ua + 1):
    c_at = random_date()
    ua_data.append({
        'id': f'UA{str(i).zfill(5)}', # UA00001
        'user_id': random.choice(student_ids),
        'assessment_id': random.choice(assessment_list),
        'score': round(random.uniform(0, 100), 2),
        'completed_at': c_at + timedelta(minutes=random.randint(10, 45)),
        'created_at': c_at,
        'updated_at': c_at + timedelta(minutes=60)
    })

df_ua = pd.DataFrame(ua_data)

# Bikin Kotor: Skor ga masuk akal, waktu selesai lebih dulu dari dibuat, ghost user
df_ua.loc[df_ua.sample(10).index, 'score'] = 999.0
df_ua.loc[df_ua.sample(10).index, 'score'] = -50.0
df_ua.loc[df_ua.sample(15).index, 'completed_at'] = df_ua.loc[df_ua.sample(15).index, 'created_at'] - timedelta(days=1)
df_ua.at[0, 'user_id'] = 'S9999' # Ghost user (User ID tidak ada di users.csv)


# --- 5. GENERATE USER PROGRESS ---
n_up = 1500
up_data = []
levels = ['beginner', 'intermediate', 'advanced']
statuses = ['not started', 'in progress', 'done']

for i in range(1, n_up + 1):
    c_at = random_date()
    up_data.append({
        'id': f'UP{str(i).zfill(5)}', # UP00001
        'user_id': random.choice(student_ids),
        'subchapter_id': random.choice(subchapter_ids), # Menggunakan ID baru
        'chapter_taken_id': random.choice(cht_ids),
        'current_level': random.choice(levels),
        'status': random.choice(statuses),
        'created_at': c_at,
        'updated_at': c_at + timedelta(days=random.randint(1, 7))
    })

df_up = pd.DataFrame(up_data)

# Bikin Kotor: Format status campur aduk, null values
df_up.loc[df_up.sample(frac=0.05).index, 'status'] = 'DONE'
df_up.loc[df_up.sample(frac=0.05).index, 'status'] = 'in-progress'
df_up.loc[df_up.sample(frac=0.05).index, 'current_level'] = np.nan


# --- 6. SAVE TO CSV ---
df_ct.to_csv('chapters_taken_dirty.csv', index=False)
df_as.to_csv('assessments_dirty.csv', index=False)
df_ua.to_csv('user_attempts_dirty.csv', index=False)
df_up.to_csv('user_progress_dirty.csv', index=False)